In [16]:
from scipy.stats import multivariate_normal
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd



In [17]:
df = pd.read_csv('embedded_system_network_security_dataset.csv')

In [18]:
df.head()

,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,mean_packet_size,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.0,0.834066,0.534891,0.0,False,True,True,False,False,False,False,False,False
1,0.527559,0.741288,59382,443,0.785714,0.0,0.147196,0.990757,0.0,False,True,False,False,False,True,False,True,False
2,0.226199,0.485116,65484,80,0.285714,0.0,0.855192,0.031781,0.0,False,True,False,False,True,False,False,False,False
3,0.573372,0.450965,51707,53,0.142857,0.0,0.153220,0.169958,0.0,False,False,False,True,False,False,False,False,False
4,0.651396,0.888740,26915,53,0.714286,0.0,0.923916,0.552053,0.0,True,False,False,True,False,False,False,True,False


In [19]:
df.tail()

,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,mean_packet_size,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
995,0.103794,0.078032,26435,443,0.428571,0.0,0.339083,0.402062,0.0,False,True,False,False,True,False,False,True,False
996,0.727989,0.048901,49767,80,0.928571,0.0,0.094822,0.418902,0.0,False,False,False,True,True,False,False,False,True
997,0.952756,0.731936,53507,80,0.928571,0.0,0.016880,0.441276,0.0,False,True,False,True,True,False,False,False,False
998,0.322835,0.655933,17255,53,0.071429,0.0,0.887533,0.806790,0.0,False,False,False,True,False,True,False,False,True
999,0.176807,0.186102,7602,80,0.857143,0.0,0.855640,0.073257,0.0,False,False,False,False,False,False,False,True,False


In [20]:
df.isnull().sum()

packet_size              0
inter_arrival_time       0
src_port                 0
dst_port                 0
packet_count_5s          0
mean_packet_size         0
spectral_entropy         0
frequency_band_energy    0
label                    0
protocol_type_TCP        0
protocol_type_UDP        0
src_ip_192.168.1.2       0
src_ip_192.168.1.3       0
dst_ip_192.168.1.5       0
dst_ip_192.168.1.6       0
tcp_flags_FIN            0
tcp_flags_SYN            0
tcp_flags_SYN-ACK        0
dtype: int64

In [24]:
features = [
    "packet_size",
    "inter_arrival_time",
    "packet_count_5s",
    "spectral_entropy",
    "frequency_band_energy"

]

normal_df = df[df['label'] == 0]
anamoly_df = df[df["label"] == 1]

normal_df.shape, anamoly_df.shape

((900, 18), (100, 18))

In [25]:
from sklearn.model_selection import train_test_split

normal_train, normal_temp = train_test_split(normal_df, test_size=0.3, random_state=42)

normal_val, normal_test = train_test_split(normal_temp, test_size=0.5, random_state=42)

anamoly_val, anamoly_test = train_test_split(anamoly_df, test_size=0.5, random_state=42)

X_train = normal_train[features].values

X_val = pd.concat([normal_val, anamoly_val])[features].values

y_val = pd.concat([normal_val, anamoly_val])['label'].values

X_test = pd.concat([normal_test, anamoly_test])[features].values

y_test = pd.concat([normal_test, anamoly_test])['label'].values
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Test set shape:", X_test.shape)

Training set shape: (630, 5)
Validation set shape: (185, 5)
Test set shape: (185, 5)


In [26]:
mu = np.mean(X_train, axis=0)
sigma2 = np.var(X_train, axis=0)

mu, sigma2

(array([0.50979878, 0.51398524, 0.52346939, 0.50012834, 0.48555378]),
 array([0.08086205, 0.08005932, 0.09028981, 0.0876324 , 0.08943072]))

In [27]:
def gaussian_probability(X, mu, sigma2):
    p = multivariate_normal.pdf(X, mean=mu, cov=sigma2)
    return p

In [29]:
p_train = gaussian_probability(X_train, mu, sigma2)
p_val = gaussian_probability(X_val, mu, sigma2)
p_test = gaussian_probability(X_test, mu, sigma2)

p_val[:10]

array([0.45540638, 1.02764161, 0.07072751, 1.49372654, 0.40347316,
       0.09970216, 0.60182525, 0.17551877, 0.20042038, 2.0209712 ])

In [31]:
epsilons = np.linspace(min(p_val), max(p_val), 1000)

best_epsilon = None
best_f1 = 0

for epsilon in epsilons:
    y_pred = (p_val < epsilon).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1> best_f1:
        best_f1 = f1
        best_epsilon = epsilon


best_epsilon, best_f1



(np.float64(1.02426043203176), 0.42857142857142855)

In [ ]:
#Evalaute on test set
